# Tuần 4 — Fine-tune Baseline 3: ViSoBERT (`uitnlp/visobert`) trên Google Colab T4

**Mục tiêu:** Huấn luyện mô hình Pretrained Transformer chuyên biệt cho văn bản mạng xã hội tiếng Việt (`uitnlp/visobert`, EMNLP 2023) với 3 random seeds (`42, 43, 44`) trên tập Benchmark công khai (`data/public/train.jsonl` & `val.jsonl`), phục vụ đối sánh **RQ1 (Domain Shift)** và **RQ2 (Ablation Normalizer)**.

- **Siêu tham số chuẩn:** `lr = 2e-5`, `batch_size = 16`, `epochs = 4`, `max_length = 160`, `weight_decay = 0.01`, `fp16 = True`.
- **Đầu ra:** Lưu checkpoint tốt nhất vào `models/visobert/`.

In [ ]:
# !pip install -q transformers datasets accelerate scikit-learn torch
import json
from pathlib import Path

import numpy as np
import torch
from sklearn.metrics import f1_score

MODEL_NAME = "uitnlp/visobert"
SEEDS = [42, 43, 44]
LABEL2ID = {"negative": 0, "neutral": 1, "positive": 2}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}
print(f"Using device: {'cuda' if torch.cuda.is_available() else 'cpu'}")

In [ ]:
def load_jsonl(path: str):
    with open(path, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {"f1_macro": float(f1_score(labels, preds, average="macro", zero_division=0))}

In [ ]:
# Kịch bản huấn luyện chuẩn với HuggingFace Trainer cho 3 seeds (42, 43, 44)
def train_visobert_multi_seed(
    train_path="data/public/train.jsonl",
    val_path="data/public/val.jsonl",
    out_dir="models/visobert",
):
    from transformers import (
        AutoModelForSequenceClassification,
        AutoTokenizer,
        Trainer,
        TrainingArguments,
        set_seed,
    )

    train_data = load_jsonl(train_path)
    val_data = load_jsonl(val_path)
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

    class ReviewDataset(torch.utils.data.Dataset):
        def __init__(self, items):
            self.enc = tokenizer(
                [x["text"] for x in items], truncation=True, padding="max_length", max_length=160
            )
            self.labels = [LABEL2ID.get(x.get("overall_sentiment", "neutral"), 1) for x in items]

        def __len__(self):
            return len(self.labels)

        def __getitem__(self, idx):
            item = {k: torch.tensor(v[idx]) for k, v in self.enc.items()}
            item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
            return item

    ds_train, ds_val = ReviewDataset(train_data), ReviewDataset(val_data)
    seed_scores = []
    best_f1 = -1.0

    for seed in SEEDS:
        set_seed(seed)
        model = AutoModelForSequenceClassification.from_pretrained(
            MODEL_NAME, num_labels=3, id2label=ID2LABEL, label2id=LABEL2ID
        )
        args = TrainingArguments(
            output_dir=f"{out_dir}_seed_{seed}",
            learning_rate=2e-5,
            per_device_train_batch_size=16,
            per_device_eval_batch_size=16,
            num_train_epochs=4,
            weight_decay=0.01,
            fp16=torch.cuda.is_available(),
            eval_strategy="epoch",
            save_strategy="epoch",
            load_best_model_at_end=True,
            metric_for_best_model="f1_macro",
            seed=seed,
            report_to="none",
        )
        trainer = Trainer(
            model=model,
            args=args,
            train_dataset=ds_train,
            eval_dataset=ds_val,
            compute_metrics=compute_metrics,
        )
        trainer.train()
        metrics = trainer.evaluate()
        f1 = metrics["eval_f1_macro"]
        seed_scores.append(f1)
        print(f"Seed {seed} -> Val Macro-F1: {f1:.4f}")
        if f1 > best_f1:
            best_f1 = f1
            Path(out_dir).mkdir(parents=True, exist_ok=True)
            model.save_pretrained(out_dir)
            tokenizer.save_pretrained(out_dir)

    print(
        f"Summary across 3 seeds: Mean F1 = {np.mean(seed_scores):.4f} ± {np.std(seed_scores):.4f}"
    )


# CreBy: MenBoyBMN